## OSM Street Number Downloader


In [1]:
import os
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import geopandas as gpd
import osmnx as ox
import pandas as pd
from pathlib import Path
import yaml

ROOT_PATH = "/Users/terezasaskova/Library/Mobile Documents/com~apple~CloudDocs/3-30-300-index"
os.chdir(ROOT_PATH)


In [2]:
from pathlib import Path
import yaml
import osmnx as ox

# Manually define the path (assuming cities.yaml is in the current directory)
_CITIES_FILE = Path.cwd() / "config.yaml"

# Load the YAML file
with open(_CITIES_FILE) as f:
    CITIES = yaml.safe_load(f)

print(f"Loaded {len(CITIES)} cities: {list(CITIES.keys())}")

city_name = 'napoli'
if city_name in CITIES:
    config = CITIES[city_name]
    gdf = ox.geocode_to_gdf(config['query'], which_result=1)


Loaded 22 cities: ['bologna', 'bari', 'bolzano', 'firenze', 'genova', 'milano', 'napoli', 'palermo', 'roma', 'torino', 'trento', 'verona', 'aosta', 'venezia', 'trieste', 'perugia', 'ancona', "l'aquila", 'campobasso', 'potenza', 'catanzaro', 'cagliari']


In [8]:
ox.settings.log_console = True
ox.settings.use_cache = True

# 3. Select City to Process
# You can change this to any key present in your YAML (e.g., 'milano', 'roma', 'bologna')
city_name = 'palermo' 

if city_name not in CITIES:
    print(f"Error: '{city_name}' not found in config.yaml. Available cities: {list(CITIES.keys())}")
    exit()

config = CITIES[city_name]
CITY = city_name  # e.g., 'napoli'
QUERY = config['query'] # e.g., 'Naples, Italy'

print(f"Processing: {CITY.upper()} (Query: {QUERY})")

# 4. Fetch Boundary from OSM
print(f"Fetching {CITY} boundary from OpenStreetMap...")
try:
    # Use the query from YAML
    city_gdf = ox.geocode_to_gdf(QUERY, which_result=1)
    
    # Ensure CRS is WGS84
    if city_gdf.crs != "EPSG:4326":
        city_gdf = city_gdf.to_crs("EPSG:4326")
        
    area = city_gdf.area.iloc[0]
    print(f"✓ {CITY} boundary loaded. Area: {area:.2f} sq units")
    
except Exception as e:
    print(f"✗ Error fetching boundary: {e}")
    print(f"Tip: Check if '{QUERY}' is correct in your YAML file.")
    exit()

Processing: PALERMO (Query: Palermo, Italy)
Fetching palermo boundary from OpenStreetMap...
✓ palermo boundary loaded. Area: 0.02 sq units


/var/folders/73/tnmscj2j2k1d5vykfvr7gp7r0000gn/T/ipykernel_51140/3068587509.py:28: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  area = city_gdf.area.iloc[0]


In [ ]:
df = pd.read_csv(f"{CITY}.csv", sep=';')

#'COORD_X_COMUNE' - Longitude 
#'COORD_Y_COMUNE' - Latitude 
x_col = 'COORD_X_COMUNE'
y_col = 'COORD_Y_COMUNE'

# Replace comma with dot and convert to float
df[x_col] = df[x_col].str.replace(',', '.').astype(float)
df[y_col] = df[y_col].str.replace(',', '.').astype(float)

# Create Geometry
# Create a list of Point objects
geometry = [Point(xy) for xy in zip(df[x_col], df[y_col])]

# Specify the Coordinate Reference System (CRS)
# WGS84 (EPSG:4326) is standard for Lat/Lon
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")


output_geojson = f"{CITY}_civic.geojson"
gdf.to_file(output_geojson, driver='GeoJSON')
print(f"Saved to {output_geojson}")


/var/folders/73/tnmscj2j2k1d5vykfvr7gp7r0000gn/T/ipykernel_43240/1038170004.py:5: DtypeWarning: Columns (3,5,6,9,12) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('campania.csv', sep=';')


Saved to Campania_civic.geojson


In [ ]:
input_file = f"{CITY}_civic.geojson"

if not Path(input_file).exists():
    print(f"Warning: Input file '{input_file}' not found. Skipping point filtering.")
    points_gdf = gpd.GeoDataFrame() # Empty dataframe
else:
    print(f"Loading address points from {input_file}...")
    points_gdf = gpd.read_file(input_file)

    if points_gdf.crs != "EPSG:4326":
        points_gdf = points_gdf.to_crs("EPSG:4326")

# 6. Filter Points Inside Boundary
if not points_gdf.empty:
    print(f"Filtering points inside {CITY}...")
    filtered_gdf = gpd.sjoin(points_gdf, city_gdf, how="inner", predicate="within")
    
    # Clean up temporary column
    if 'index_right' in filtered_gdf.columns:
        filtered_gdf = filtered_gdf.drop(columns=['index_right'])
        
    print(f"Total points loaded: {len(points_gdf)}")
    print(f"Points inside {CITY}: {len(filtered_gdf)}")
    print(f"Points outside {CITY}: {len(points_gdf) - len(filtered_gdf)}")
else:
    filtered_gdf = gpd.GeoDataFrame()
    print("No points to filter.")

output_dir = Path("data")
output_dir.mkdir(exist_ok=True)


Loading address points from napoli_civic.geojson...
Filtering points inside napoli...
Total points loaded: 2133834
Points inside napoli: 126934
Points outside napoli: 2006900


In [ ]:
output_geojson = f"{CITY}_buildings_streetnum.geojson"
filtered_gdf.to_file(output_geojson, driver='GeoJSON')
print(f"Saved to {output_geojson}")